# Práctica: Outliers en Mall_Customers con KDE y LOF

Datos:
- Annual Income (k$)
- Spending Score (1-100)

Métodos:
- KDE: outlier si densidad < percentil 10
- LOF: comparar contamination=0.1 vs 'auto'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity, LocalOutlierFactor

## 1) Carga escalado

In [ ]:
df = pd.read_csv("../data/Mall_Customers.csv") # funciona solo con el csv descargado

cols = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[cols].copy()

print("Filas:", len(df))
display(X.head())

scaler = StandardScaler()
X_s = scaler.fit_transform(X.values)

print("Media (aprox):", X_s.mean(axis=0))
print("Std (aprox):", X_s.std(axis=0))

## 2) Grid 2D para dibujar densidades

In [ ]:
def make_grid(X_scaled, step=0.02, pad=0.6):
    x_min, x_max = X_scaled[:, 0].min() - pad, X_scaled[:, 0].max() + pad
    y_min, y_max = X_scaled[:, 1].min() - pad, X_scaled[:, 1].max() + pad

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, step),
        np.arange(y_min, y_max, step)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    return xx, yy, grid

xx, yy, grid = make_grid(X_s)

## 3) KDE
Parámetros:
- bandwidth: [0.2, 0.5, 0.8]
- kernel: gaussian, tophat

Regla:
- outlier si densidad(x) < percentil 10 de densidad

In [ ]:
bandwidths = [0.2, 0.5, 0.8]
kernels = ["gaussian", "tophat"]

def kde_outliers(X_scaled, bandwidth, kernel, percentile=10):
    kde = KernelDensity(bandwidth=bandwidth, kernel=kernel)
    kde.fit(X_scaled)

    # densidad en los puntos del dataset
    log_d = kde.score_samples(X_scaled)
    d = np.exp(log_d)
    thr = np.percentile(d, percentile)
    out = d < thr

    # densidad en el grid
    log_d_grid = kde.score_samples(grid)
    d_grid = np.exp(log_d_grid).reshape(xx.shape)

    return out, thr, d_grid

In [ ]:
kde_rows = []

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
idx = 0

for ker in kernels:
    for bw in bandwidths:
        out, thr, d_grid = kde_outliers(X_s, bw, ker, percentile=10)

        kde_rows.append({
            "kernel": ker,
            "bandwidth": bw,
            "threshold_p10": float(thr),
            "n_outliers": int(out.sum())
        })

        ax = axes[idx]
        ax.contourf(xx, yy, d_grid, levels=30, alpha=0.6)
        ax.scatter(X_s[~out, 0], X_s[~out, 1], s=16, c="blue")
        ax.scatter(X_s[out, 0], X_s[out, 1], s=18, c="red")
        ax.set_title(f"KDE {ker}, bw={bw}")
        ax.set_xlabel("Income (scaled)")
        ax.set_ylabel("Spending (scaled)")
        idx += 1

plt.tight_layout()
plt.show()

df_kde = pd.DataFrame(kde_rows)
display(df_kde)

## 4) LOF
Parámetros:
- n_neighbors: [20, 35, 50]
- comparar contamination=0.1 y contamination='auto'

In [ ]:
neighbors_list = [20, 35, 50]

def lof_outliers(X_scaled, n_neighbors, contamination):
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
    y_hat = lof.fit_predict(X_scaled)  # -1 outlier, 1 normal
    return (y_hat == -1)

In [ ]:
lof_rows = []

# Figura 1: contamination = 0.1
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, nn in enumerate(neighbors_list):
    out = lof_outliers(X_s, nn, contamination=0.1)
    lof_rows.append({"contamination": 0.1, "n_neighbors": nn, "n_outliers": int(out.sum())})

    ax = axes[i]
    ax.scatter(X_s[~out, 0], X_s[~out, 1], s=16, c="blue")
    ax.scatter(X_s[out, 0], X_s[out, 1], s=18, c="red")
    ax.set_title(f"LOF cont=0.1, n={nn}")
    ax.set_xlabel("Income (scaled)")
    ax.set_ylabel("Spending (scaled)")
plt.tight_layout()
plt.show()

# Figura 2: contamination = auto
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, nn in enumerate(neighbors_list):
    out = lof_outliers(X_s, nn, contamination="auto")
    lof_rows.append({"contamination": "auto", "n_neighbors": nn, "n_outliers": int(out.sum())})

    ax = axes[i]
    ax.scatter(X_s[~out, 0], X_s[~out, 1], s=16, c="blue")
    ax.scatter(X_s[out, 0], X_s[out, 1], s=18, c="red")
    ax.set_title(f"LOF cont=auto, n={nn}")
    ax.set_xlabel("Income (scaled)")
    ax.set_ylabel("Spending (scaled)")
plt.tight_layout()
plt.show()

df_lof = pd.DataFrame(lof_rows)
display(df_lof)

## 5) Resultado

KDE (umbral = percentil 10 de densidad):
- Con kernel gaussian salen 20 outliers en los tres bandwidth (0.2, 0.5, 0.8). Es lo esperado al usar el percentil 10 en 200 puntos.
- Con kernel tophat el número cambia: bw = 0.2 da 0 outliers, bw = 0.5 da 15 y bw = 0.8 da 18. Con kernel caja aparecen muchos empates de densidad y, al usar la condición densidad < umbral, algunos puntos quedan justo en el umbral y no se marcan.

LOF:
- Con contamination = 0.1 salen siempre 20 outliers para n_neighbors = 20, 35 y 50 (forzamos el 10%).
- Con contamination = auto varía mucho: n_neighbors=20 da 9 outliers, n_neighbors = 35 da 43 y n_neighbors=50 da 65. Aquí el umbral lo decide el método y es muy sensible a n_neighbors.

Conclusión:
- KDE es global (baja densidad en el plano) y depende mucho de bandwidth y kernel.
- LOF es local (comparación con vecinos) y depende mucho de n_neighbors, sobre todo si contamination no está fijado.